# *RAG Chatbot – From Scratch*

### Initial Steps
---
- Import the libraries
- Load the environment variables
- Define an LLM model

In [1]:
# Import necessary libraries
from dotenv import load_dotenv
from langchain_huggingface import HuggingFaceEndpoint
import warnings
from langchain_openai import ChatOpenAI
from langchain.prompts.prompt import PromptTemplate
import os

warnings.filterwarnings("ignore")

C:\Users\Zeynep\Desktop\DS\ds-rag-pipeline\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# Load environment variables
load_dotenv()

True

In [3]:
# Define the LLM model
llm = ChatOpenAI(
    model="meta-llama/llama-3.1-8b-instruct",
    api_key=os.environ.get("OPENAI_API_KEY"),
    base_url="https://openrouter.ai/api/v1",
    temperature=0
)

### 1. Data Ingestion

---

Load the PDF and extract raw content
1. Import PDF Loader
2. Load PDF File


In [4]:
# Import the PDF loader
from langchain_community.document_loaders import PyPDFLoader

In [5]:
# Define a function to load PDFs
def load_pdf(pdf_path):
    loader = PyPDFLoader(file_path=pdf_path)
    documents = loader.load()
    return documents

In [6]:
# Load the pdf file
react_docs = load_pdf(pdf_path = '../documents/SustainableFashion-SON.pdf')

In [7]:
print(f'Number of pages in the article: {len(react_docs)}')

Number of pages in the article: 16


In [8]:
print(f'First page of the article: {react_docs[0].page_content}')

First page of the article: Journal of Innovations and Sustainability 
ISSN 2367-8151 
2024, Vol. 8, No. 3 
https://is-journal.com 
 
 
ANALYSIS OF CONSUMER PREFERENCES IN SUSTAINABLE FASHION 
CONSUMPTION 
 
Zeynep Bumin 
Humboldt Universitat zu Berlin, Germany 
ORCID: https://orcid.org/0009-0004-2890-0374 
 
Mete Bumin 
Banking Regulation and Supervision Agency, Türkiye 
ORCID: https://orcid.org/0000-0002-4740-0007 
 
 
Bumin, Z., & Bumin,  M. (2024). Analysis of consumer preferences in sustainable fashion 
consumption. Journal of Innovations and Sustainability , 8(3), 01.  
https://doi.org/10.51599/is.2024.08.03.01. 
 
Purpose. Growing consumer concern about the environmental and ethical impact of their clothing 
purchases has led to an increased desire for more conscious alternatives, which has stimulated the 
emergence of a market for products that prioritise sustainability. This study aims to explore the 
complexities of consumer preferences and choices in sustainable fashion consu

### 2. Chunking 
---
Split the document into meaningful text blocks

1. Import text splitter
2. Chunk the documents

---
**What chunk size and overlap makes sense for your document?**

We are splitting the 16-page PDF into chunks of 600 tokens with a 100-token overlap. This size is large enough to keep related ideas together in a single block while the overlap ensures no context is lost at the boundaries.

In [10]:
# Import the text splitter
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [11]:
# Create a chunking function
def chunk_documents(documents, chunk_size=600, chunk_overlap=100):
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        separators=["\n\n", "\n", " ", ""] # Priority: Paragraphs -> Lines -> Words
    )
    
    chunks = text_splitter.split_documents(documents=documents)
    
    for i, chunk in enumerate(chunks):
        # Extract filename from metadata if it exists
        source_name = chunk.metadata.get("source", "doc").split("/")[-1]
        chunk.metadata.update({
            "id": f"{source_name}_chunk_{i}",
            "chunk_index": i
        })
    
    return chunks

In [12]:
react_chunks = chunk_documents(react_docs)

In [13]:
print(f"Number of chunks created: {len(react_chunks)}")

Number of chunks created: 101


In [14]:
react_chunks[5]

Document(metadata={'producer': 'Microsoft® Word 2019', 'creator': 'Microsoft® Word 2019', 'creationdate': '2024-10-17T14:52:38+03:00', 'author': 'Admin', 'moddate': '2024-10-17T14:52:38+03:00', 'source': '../documents/SustainableFashion-SON.pdf', 'total_pages': 16, 'page': 0, 'page_label': '1', 'id': 'SustainableFashion-SON.pdf_chunk_5', 'chunk_index': 5}, page_content='on choosing sustainable clothing by quantifying the comparative significance of the preferences o f \ncustomers. \nPractical value. This study aspires to contribute a fresh perspective and valuable insights to the field \nof sustainable fashion. Furthermore, it intends to offer practical recommendations for brands and \npolicymakers as they navigate this ever -evolving landscape. Recogni sing the determinants of \nconsumers’ intentions to purchase sustainable garments is imperative for fashion brands and')

### 3. Embeddings
---
Convert chunks into vectors and store them
1. Choose an embedding model
2. Transform the document chunks to embeddings
3. Store the embeddings created

---
**Which embedding model do you choose, and why?**

While all-mpnet-base-v2 is a classic local baseline, I am using text-embedding-3-small for its superior 8k context window and higher retrieval accuracy. This ensures our 600-character chunks are fully processed without truncation, leading to more reliable answers from the chatbot.

It's important to note that:
- The OpenAI API is designed to return already normalized vectors by default.
- The OpenAIEmbeddings class in LangChain handles the formatting so that the resulting vectors are perfectly ready for Cosine Similarity without you needing to add extra lines of code.

In [15]:
from langchain_openai import OpenAIEmbeddings
from langchain_community.vectorstores import FAISS

In [30]:
def embed_and_store(chunks, db_name):
    embedding = OpenAIEmbeddings(
        model="text-embedding-3-small",
        base_url="https://openrouter.ai/api/v1"
    )

    # FAISS will use these normalized vectors to calculate 
    # similarity accurately without extra manual steps.
    vectorstore = FAISS.from_documents(
        documents=chunks,
        embedding=embedding
    )

    vectorstore.save_local(f"../vector_databases/vector_db_{db_name}")
    return vectorstore

In [31]:
# apply the embeddings and store them
all_embedding = embed_and_store(chunks=react_chunks, db_name='react')

### 4. Retrieval
---
Retrieve relevant chunks for a given question
1. Load internal stored embeddings
2. Implement retrieval logic
3. Test it with a query

---
**How many chunks do you retrieve per query (k)?**

$k=4$ is set for retrieval. This value ensures the chatbot receives a comprehensive set of facts from the 16-page PDF while remaining within the model's optimal context window, preventing irrelevant information from degrading the answer quality.

In [32]:
# define a retrieval function from internally stored embeddings
def retrieve_from_vector_db(vector_db_path, k):

    embeddings = OpenAIEmbeddings(
        model="text-embedding-3-small",
        base_url="https://openrouter.ai/api/v1"
    )

    # OpenAI is normalized by default, so don't need to define a distance strategy here
    vectorstore = FAISS.load_local(
        folder_path=vector_db_path,
        embeddings=embeddings,
        allow_dangerous_deserialization=True
    )
    
    retriever = vectorstore.as_retriever(search_kwargs={"k": k}) 
    
    return retriever, vectorstore

In [33]:
# implement the retrieval logic
retriever, vectorstore = retrieve_from_vector_db("../vector_databases/vector_db_react", k=4)

In [35]:
# test the logic with a query
test_query = 'concept of sustainable fashion'

In [36]:
retriever.invoke(test_query, k=4)

[Document(id='56091922-1970-4db5-94e5-b957e9f8c92f', metadata={'producer': 'Microsoft® Word 2019', 'creator': 'Microsoft® Word 2019', 'creationdate': '2024-10-17T14:52:38+03:00', 'author': 'Admin', 'moddate': '2024-10-17T14:52:38+03:00', 'source': '../documents/SustainableFashion-SON.pdf', 'total_pages': 16, 'page': 1, 'page_label': '2', 'id': 'SustainableFashion-SON.pdf_chunk_9', 'chunk_index': 9}, page_content='impact of their clothing purchases has led to a heightened desire for more conscious  \nalternatives. This, in turn, has created a market for products that prioriti se \nsustainability.  \nThis study aims to explore the complexities of consumer preferences and choices \nin sustainable fashion consumption by uncovering the key factors that guide consumers \nin selecting clothing items. The ultimate objective is to quantify the comparative \nsignificance of attributes and provide insights into the factors that influence consumers'),
 Document(id='6cdd687a-97ef-40d9-afab-75014c88

### 5. LLM Integration
---
Generate an answer based on the retrieved chunks
1. Connect document retrieval with the Language Model
2. Test by generating answers from retrieved document

---
**How do you formulate the prompt to the LLM?**

Formulating the prompt is the process of combining the retrieved context chunks and the user's question into a single structured instruction for the LLM. This ensures the model's response is grounded in the specific data from the PDF rather than its general training data.

In [38]:
from langchain.chains.combine_documents import create_stuff_documents_chain
from langchain.chains import create_retrieval_chain
from langchain import hub

In [47]:
# create a retrieval chain
def connect_chains(retriever):
    
    stuff_documents_chain = create_stuff_documents_chain(
        llm=llm,
        prompt=hub.pull("langchain-ai/retrieval-qa-chat")
    )
    
    retrieval_chain = create_retrieval_chain(
        retriever=retriever,
        combine_docs_chain=stuff_documents_chain
    )
    
    return retrieval_chain

retrieval_chain = connect_chains(retriever)

In [48]:
# invoke the chain with a sample question
output = retrieval_chain.invoke(
    {'input': 'what is sustainable fashion?'}
)

type(output), output.keys()

(dict, dict_keys(['input', 'context', 'answer']))

In [49]:
# k = 4 chunks retrieved per query
output['context']

[Document(id='56091922-1970-4db5-94e5-b957e9f8c92f', metadata={'producer': 'Microsoft® Word 2019', 'creator': 'Microsoft® Word 2019', 'creationdate': '2024-10-17T14:52:38+03:00', 'author': 'Admin', 'moddate': '2024-10-17T14:52:38+03:00', 'source': '../documents/SustainableFashion-SON.pdf', 'total_pages': 16, 'page': 1, 'page_label': '2', 'id': 'SustainableFashion-SON.pdf_chunk_9', 'chunk_index': 9}, page_content='impact of their clothing purchases has led to a heightened desire for more conscious  \nalternatives. This, in turn, has created a market for products that prioriti se \nsustainability.  \nThis study aims to explore the complexities of consumer preferences and choices \nin sustainable fashion consumption by uncovering the key factors that guide consumers \nin selecting clothing items. The ultimate objective is to quantify the comparative \nsignificance of attributes and provide insights into the factors that influence consumers'),
 Document(id='6cdd687a-97ef-40d9-afab-75014c88

In [51]:
# optimal answer
output['answer']

'Based on the context, sustainable fashion refers to clothing items that prioritize environmental and ethical considerations. It implies that the production, manufacturing, and distribution of these clothing items have a reduced impact on the environment and adhere to ethical standards.'

### 5. Chat Function
---
Simple interactive query loop
1. Create chat interface by developing a function for interactive querying
2. Run the interactive chat for immediate responses

In [52]:
# define the interface of interactive chat querying
def chat_with_rag(chain):
    print("Welcome to the RAG Chat! Type 'exit' to quit.\n")
    while True:
        user_input = input("🧑 You: ")
        if user_input.lower() in ["exit", "quit"]:
            print("👋 Exiting the chat. Goodbye!")
            break
        try:
            result = chain.invoke({"input": user_input})
            print(f"🤖 RAG Answer: {result['answer']}\n")
        except Exception as e:
            print(f" Error: {e}\n")

In [ ]:
# start the chat
chat_with_rag(retrieval_chain)

Welcome to the RAG Chat! Type 'exit' to quit.



🧑 You:  How can we make more sustainable clothing choices?


🤖 RAG Answer: Based on the context, here are some key takeaways on how to make more sustainable clothing choices:

1. **Choose products made from high-quality materials**: Consumers prioritize material quality and environmental impact when making sustainable fashion choices.
2. **Opt for products with ethical production practices**: Ethical labor conditions are a crucial factor in shaping consumer decision-making.
3. **Consider the brand's reputation**: While not the most important factor, brand reputation is still a consideration for consumers.
4. **Look for sustainability certificates**: While not as decisive as other factors, sustainability certificates can still be a useful indicator of a product's environmental impact.
5. **Be mindful of price**: Price is a significant factor in consumer decision-making, so be prepared to pay a bit more for sustainable clothing options.

By considering these factors, consumers can make more informed and sustainable clothing choices that align with

🧑 You:  Which model used to understand consumer preferences?


🤖 RAG Answer: Conjoint analysis is the model used to understand consumer preferences.



...

🤖 RAG Answer: The target group of the study appears to be a relatively young, well-educated, and predominantly female demographic. The study mentions that the sample is skewed towards a younger demographic and that the majority of respondents are students, which suggests that the target group is likely university students or individuals in their early to mid-twenties.



🧑 You:  Can you summarize the top 5 learnings briefly?


🤖 RAG Answer: Unfortunately, the provided context doesn't explicitly mention the top 5 learnings. However, I can infer some key points based on the text:

1. **Importance of attributes**: The study explores the importance of various attributes in consumers' purchasing decisions, such as price, quality, and supply chain transparency.
2. **Percentage-based importance**: The study uses percentage-based importance to simplify the communication of results and facilitate direct and universal comparisons.
3. **Key attributes**: The study identifies key attributes that exert significant influence on consumers' choices, such as labor conditions and supply chain transparency.
4. **Real-life decision scenarios**: The study incorporates real-life decision scenarios, such as price, to increase the results' relevance and accurately reflect consumer behavior.
5. **Universal comparisons**: The study's findings can be compared universally, allowing for a deeper understanding of consumer behavior and de

🧑 You:  How to consume fashion more ethically?


🤖 RAG Answer: Based on the context, here are some tips on how to consume fashion more ethically:

1. **Prioritize sustainability**: Look for products that prioritize sustainability, such as those made from eco-friendly materials, produced with minimal waste, and designed for longevity.
2. **Consider the production process**: Be aware of the production process and the potential harm it may cause to workers, the environment, and the community. Opt for products made with fair labor practices and minimal harm to the environment.
3. **Check for sustainability certifications**: Look for certifications such as those mentioned in the context (e.g., Gavranovic, 2020) that ensure the product meets certain sustainability standards.
4. **Be mindful of production location**: Consider the distance between where the product is produced and delivered to consumers, as well as the potential exploitation of labor in certain regions.
5. **Make conscious choices**: Be aware of the multiple considerations i